In [23]:
import os
from dotenv import load_dotenv
from huggingface_hub import InferenceClient

load_dotenv()
token = os.getenv("HF_TOKEN")

# We initialize the client. This automatically handles the Router URLs for us.
client = InferenceClient(api_key=token)

print("✅ Official HF Inference Client initialized.")

/home/shoaib/Desktop/DHC Internship/Task_04_Health_Chatbot/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Official HF Inference Client initialized.


In [ ]:
def health_assistant(user_query):
    # 1. LOCAL SAFETY FILTER
    danger_keywords = ["suicide", "harm", "kill myself", "overdose", "emergency"]
    if any(word in user_query.lower() for word in danger_keywords):
        return "🚨 [SAFETY ALERT]: Please contact emergency services (911) or a crisis hotline immediately."

    # 2. EXECUTION VIA SDK
    try:
        # Using Llama-3.2-1B-Instruct: A dedicated Chat model
        response = client.chat.completions.create(
            model="meta-llama/Llama-3.2-1B-Instruct", 
            messages=[
                {
                    "role": "system", 
                    "content": (
                        "You are a professional medical assistant. "
                        "MANDATORY: Your first sentence must be 'I am an AI assistant, not a doctor.' "
                        "Provide general health info. If asked about kids' medicine, suggest a pediatrician."
                    )
                },
                {"role": "user", "content": user_query}
            ],
            max_tokens=500,
            temperature=0.3
        )
        
        return response.choices[0].message.content.strip()

    except Exception as e:
        return f"❌ Connection Error: {str(e)}"

# --- TEST RUN ---
print("--- Result 1: Sore Throat ---")
print(health_assistant("What causes a sore throat?"))

print("\n" + "="*50 + "\n")

print("--- Result 2: Kids Medicine ---")
print(health_assistant("Is paracetamol safe for a 2-year-old?"))

--- Result 1: Sore Throat ---
❌ Connection Error: (Request ID: Root=1-69a35c21-2a9744b31167c56c6e438605;c029be7f-1c6c-4edc-9e85-f2d68ac37f5f)

Bad request:
{'message': "The requested model 'mistralai/Mistral-7B-Instruct-v0.3' is not a chat model.", 'type': 'invalid_request_error', 'param': 'model', 'code': 'model_not_supported'}


--- Result 2: Kids Medicine ---
❌ Connection Error: (Request ID: Root=1-69a35c22-27713a1b3a9e18785588e1db;c787de1c-82b8-4917-a24d-afad795c9dcf)

Bad request:
{'message': "The requested model 'mistralai/Mistral-7B-Instruct-v0.3' is not a chat model.", 'type': 'invalid_request_error', 'param': 'model', 'code': 'model_not_supported'}
